# CS 228 checkpoint: data quality, EDA, baseline with threshold analysis (hands-off)

Reproduces every number in the checkpoint report. **Inputs (Add Input):** dataset `celeba-spoof-for-face-antispoofing`
(Datasets tab) and your notebook `cs228-data` (Notebooks tab). **Session options:** GPU, Internet On.
**Save Version -> Save & Run All (Commit).** About 20-25 GPU minutes. Output: `bundle_checkpoint.zip`.

In [ ]:
# 1. Get the code
%cd /kaggle/working
!rm -rf cs228-pad-project
GITHUB_USER = "ShivamShelar101"
!git clone https://github.com/{GITHUB_USER}/cs228-pad-project.git
%cd cs228-pad-project
!pip install -q onnxscript matplotlib
!git log --oneline -1

In [ ]:
# 2. Dataset location (attach it first: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing)
import glob, os
hits = []
for depth in range(1, 4):   # shallow search only: never walk the 750k-file dataset
    hits += glob.glob("/kaggle/input/" + "*/" * (depth - 1) + "celeba-spoof-for-face-antispoofing")
print("found:", hits)
assert hits, "Dataset not attached: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing"
SRC = hits[0]
print("dataset at:", SRC)

In [ ]:
# 3. Data: reuse the prebuilt data_pad2 from the attached cs228-data notebook, else build it
found = []
for depth in range(1, 6):
    found += glob.glob("/kaggle/input/" + "*/" * (depth - 1) + "data_pad2")
print("prebuilt data:", found)
if found:
    !ln -sfn {found[0]} ./data_pad2
else:
    !python -m data.prepare_celeba_spoof --src {SRC} --dst ./data_pad2 --per_class 3000 --test_per_class 600 --max_per_subject 8
!for s in train val test; do for c in live spoof; do echo $s/$c $(ls data_pad2/$s/$c | wc -l); done; done

In [ ]:
# 4. Baseline: ResNet-18 (ImageNet weights), 5 epochs, three fixed seeds
for s in [0, 1, 2]:
    !python -m training.train_baseline --data_root ./data_pad2 --epochs 5 --seed {s} --out checkpoints/baseline

In [ ]:
# 5. Threshold analysis: counts for every rate, EER, FAR/FRR, ROC, operating point chosen on validation
!python -m evaluation.threshold_analysis --data_root ./data_pad2 --ckpt_glob "checkpoints/baseline/seed*_best.pt" --out results/threshold

In [ ]:
# 6. Data quality checks + exploratory data analysis (figures, summary, captions)
!python -m evaluation.eda --src {SRC} --data_root ./data_pad2 --out results/eda

In [ ]:
# 7. Error analysis: which kinds of fakes the baseline accepts (seed 0)
!python -m evaluation.by_spoof_type --src {SRC} --data_root ./data_pad2 --ckpt checkpoints/baseline/seed0_best.pt | tee results/by_spoof_type_seed0.txt

In [ ]:
# 8. Record the commit these numbers come from, then pack everything
!git rev-parse HEAD | tee results/COMMIT.txt
!zip -qr bundle_checkpoint.zip results checkpoints/baseline/seed0_best.pt checkpoints/baseline/seed1_best.pt checkpoints/baseline/seed2_best.pt
!ls -lh bundle_checkpoint.zip